In [ ]:
# Segment tree saplings and green wet and dry reference surface from background with Segment Anything 2 using selected points from shapefiles.
# Segment per light condition (shady/sunny).
# Create plots with segmentation results

In [ ]:
# Installation SAM2

# git clone https://github.com/facebookresearch/sam2.git && cd sam2
# pip install -e .


In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import geopandas as gpd
import re
from affine import Affine
import torch
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from collections import defaultdict

In [ ]:
# Tree_Shade

def world_to_pixel(transform, x, y):
    col, row = ~transform * (x, y)
    return int(col), int(row)

def process_folder_pair(folder_path, shape_path):
    processed_folder = os.path.join(folder_path, "Processed_new")
    if not os.path.exists(processed_folder):
        print(f"Processed_new folder not found in {folder_path}")
        return

    match = re.search(r'20\d{6}', folder_path)
    date_str = match.group(0) if match else 'unknown_date'

    base_results_dir = r"D:/Bewaesserung_Forstkulturen/Daten/Gewaechshaus/Gewaechshaus_Trockenstress2023/Thermal/GH_Analyse_Thermal/segmentation_results"
    folder_name_cleaned = os.path.basename(os.path.normpath(folder_path)).replace(date_str + "_", "")
    results_subdir = f'{date_str}_{folder_name_cleaned}_Tree_Shade'
    results_path = os.path.join(base_results_dir, results_subdir)
    os.makedirs(results_path, exist_ok=True)

    checkpoint = "checkpoints/sam2.1_hiera_large.pt"
    model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"
    predictor = SAM2ImagePredictor(build_sam2(model_cfg, checkpoint))

    mask_channels = ['channel1', 'channel2', 'channel3', 'combined']
    channel_paths = {}
    for channel in mask_channels:
        path = os.path.join(results_path, channel)
        os.makedirs(path, exist_ok=True)
        channel_paths[channel] = path

    for filename in os.listdir(processed_folder):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            img_path = os.path.join(processed_folder, filename)
            print(f"Loading image: {img_path}")
            img = Image.open(img_path)

            basename = filename.replace("_thermal_rgb_transformed.jpg", "")
            matching_shapefiles = [f for f in os.listdir(shape_path) if f.startswith(basename) and f.endswith("_Points.shp")]

            if not matching_shapefiles:
                print(f"No shapefile found for basename: {basename}")
                continue
            elif len(matching_shapefiles) > 1:
                print(f"Multiple shapefiles found for basename {basename}, using the first: {matching_shapefiles[0]}")

            shape_name = os.path.join(shape_path, matching_shapefiles[0])
            gdf = gpd.read_file(shape_name)
            print(f"Number of points in the shapefile: {len(gdf)}")

            width, height = img.size
            xmin, xmax = 0, 480
            ymin, ymax = -640, 0
            xres = (xmax - xmin) / width
            yres = (ymax - ymin) / height
            transform = Affine.translation(xmin, ymax) * Affine.scale(xres, -yres)

            class_tree, class_other, class_negative = [], [], []
            tree_x, tree_y = [], []
            tree_sun_x, tree_sun_y = [], []   
            other_x, other_y = [], []

            for _, row in gdf.iterrows():
                # Check for valid geometry
                if row.geometry is None or row.geometry.is_empty:
                    print("Skipped empty or invalid geometry.")
                    continue
                if row.geometry.geom_type != 'Point':
                    print(f"Unsupported geometry type: {row.geometry.geom_type}. Skipped.")
                    continue
            
                label = str(row["Position"]).strip().lower()
                x, y = row.geometry.x, row.geometry.y
                col, row_px = world_to_pixel(transform, x, y)
            
                if "tree_shade" in label:
                    class_tree.append((col, row_px))
                    tree_x.append(col)
                    tree_y.append(row_px)
                elif "tree_sun" in label:
                    class_negative.append((col, row_px))
                    tree_sun_x.append(col)                
                    tree_sun_y.append(row_px)
                elif "green_wet_shade" in label or "green_dry_shade" in label:
                    class_other.append((col, row_px))
                    other_x.append(col)
                    other_y.append(row_px)


            img_np = np.asarray(img, dtype='float32') / 255

            fig, axs = plt.subplots(1, 5, figsize=(30, 10))
            axs[0].imshow(img_np)
            axs[0].scatter(tree_x, tree_y, c='red', marker='o', label='Tree_Shade')
            axs[0].scatter(tree_sun_x, tree_sun_y, c='purple', marker='o', label='Tree_Sun') 
            axs[0].scatter(other_x, other_y, c='blue', marker='x', label='Other')
            axs[0].legend()
            axs[0].set_title('Original Image with points')

            if class_tree:
                point_coords = np.array(class_tree)
                point_labels = np.ones(len(point_coords))

                if class_negative:
                    point_coords = np.concatenate((point_coords, np.array(class_negative)), axis=0)
                    point_labels = np.concatenate((point_labels, np.zeros(len(class_negative))))

                with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
                    predictor.set_image(img_np)
                    mask, _, _ = predictor.predict(point_coords=point_coords, point_labels=point_labels)
                mask = np.moveaxis(mask, 0, -1)

                mask1 = np.expand_dims(mask[:, :, 0], axis=-1)
                mask2 = np.expand_dims(mask[:, :, 1], axis=-1)
                mask3 = np.expand_dims(mask[:, :, 2], axis=-1)
                mask4 = np.expand_dims(mask1[:, :, 0] + mask2[:, :, 0] + mask3[:, :, 0], axis=-1)
                mask4[mask4 > 1] = 1

                out_image1 = np.multiply(mask1, img_np)
                out_image2 = np.multiply(mask2, img_np)
                out_image3 = np.multiply(mask3, img_np)
                out_image4 = np.multiply(mask4, img_np)

                axs[1].imshow(out_image1)
                axs[2].imshow(out_image2)
                axs[3].imshow(out_image3)
                axs[4].imshow(out_image4)

                axs[1].set_title('1st Channel Mask')
                axs[2].set_title('2nd Channel Mask')
                axs[3].set_title('3rd Channel Mask')
                axs[4].set_title('Combined Mask')

                Image.fromarray((out_image1 * 255).astype(np.uint8)).save(os.path.join(channel_paths['channel1'], filename))
                Image.fromarray((out_image2 * 255).astype(np.uint8)).save(os.path.join(channel_paths['channel2'], filename))
                Image.fromarray((out_image3 * 255).astype(np.uint8)).save(os.path.join(channel_paths['channel3'], filename))
                Image.fromarray((out_image4 * 255).astype(np.uint8)).save(os.path.join(channel_paths['combined'], filename))
            else:
                print("No 'Tree_Shade'-Points. Segmentation skipped.")

            plt.tight_layout()
            plt.savefig(os.path.join(results_path, filename) + '_overview.png')
            plt.show()


# ----------- Use all folders ------------------

base_dir = r"D:/Bewaesserung_Forstkulturen/Daten/Gewaechshaus/Gewaechshaus_Trockenstress2023/Thermal/GH_Analyse_Thermal/"

thermal_folders = []
shapefile_folders = []

# Search for all relevant folders
for entry in os.listdir(base_dir):
    full_path = os.path.join(base_dir, entry)
    if os.path.isdir(full_path):
        if entry.endswith("_Thermal_sorted"):
            thermal_folders.append(full_path)
        elif entry.endswith("_Shapefiles"):
            shapefile_folders.append(full_path)

# Use the corresponding shapefile folder for every folder with thermal images
for thermal_folder in thermal_folders:
    match = re.search(r'20\d{6}', thermal_folder)
    date_str = match.group(0) if match else None
    if not date_str:
        print(f"No date found in foldername: {thermal_folder}")
        continue

    matching_shapes = [sf for sf in shapefile_folders if date_str in sf]
    if not matching_shapes:
        print(f"No shapefile folder found for date {date_str}")
        continue
    if len(matching_shapes) > 1:
        print(f"Several shapefile folder found for date {date_str}, use first one: {matching_shapes[0]}")

    shape_folder = matching_shapes[0]
    process_folder_pair(thermal_folder, shape_folder)

In [ ]:
# Repeat for Tree_Sun

def world_to_pixel(transform, x, y):
    col, row = ~transform * (x, y)
    return int(col), int(row)

def process_folder_pair(folder_path, shape_path):
    processed_folder = os.path.join(folder_path, "Processed_new")
    if not os.path.exists(processed_folder):
        print(f"Processed_new folder not found in {folder_path}")
        return

    match = re.search(r'20\d{6}', folder_path)
    date_str = match.group(0) if match else 'unknown_date'

    base_results_dir = r"D:/Bewaesserung_Forstkulturen/Daten/Gewaechshaus/Gewaechshaus_Trockenstress2023/Thermal/GH_Analyse_Thermal/segmentation_results"
    folder_name_cleaned = os.path.basename(os.path.normpath(folder_path)).replace(date_str + "_", "")
    results_subdir = f'{date_str}_{folder_name_cleaned}_Tree_Sun'
    results_path = os.path.join(base_results_dir, results_subdir)
    os.makedirs(results_path, exist_ok=True)

    checkpoint = "checkpoints/sam2.1_hiera_large.pt"
    model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"
    predictor = SAM2ImagePredictor(build_sam2(model_cfg, checkpoint))

    mask_channels = ['channel1', 'channel2', 'channel3', 'combined']
    channel_paths = {}
    for channel in mask_channels:
        path = os.path.join(results_path, channel)
        os.makedirs(path, exist_ok=True)
        channel_paths[channel] = path

    for filename in os.listdir(processed_folder):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            img_path = os.path.join(processed_folder, filename)
            print(f"Loading image: {img_path}")
            img = Image.open(img_path)

            basename = filename.replace("_thermal_rgb_transformed.jpg", "")
            matching_shapefiles = [f for f in os.listdir(shape_path) if f.startswith(basename) and f.endswith("_Points.shp")]

            if not matching_shapefiles:
                print(f"No shapefile found for basename: {basename}")
                continue
            elif len(matching_shapefiles) > 1:
                print(f"Multiple shapefiles found for basename {basename}, using the first: {matching_shapefiles[0]}")

            shape_name = os.path.join(shape_path, matching_shapefiles[0])
            gdf = gpd.read_file(shape_name)
            print(f"Number of points in the shapefile: {len(gdf)}")

            width, height = img.size
            xmin, xmax = 0, 480
            ymin, ymax = -640, 0
            xres = (xmax - xmin) / width
            yres = (ymax - ymin) / height
            transform = Affine.translation(xmin, ymax) * Affine.scale(xres, -yres)

            class_tree, class_other = [], []
            class_negative = []
            tree_x, tree_y = [], []
            tree_shade_x, tree_shade_y = [], [] 
            other_x, other_y = [], []

            for _, row in gdf.iterrows():
                # Check for valid geometry
                if row.geometry is None or row.geometry.is_empty:
                    print("Skipped empty or invalid geometry.")
                    continue
                if row.geometry.geom_type != 'Point':
                    print(f"Unsupported geometry type: {row.geometry.geom_type}. Skipped.")
                    continue
            
                label = str(row["Position"]).strip().lower()
                x, y = row.geometry.x, row.geometry.y
                col, row_px = world_to_pixel(transform, x, y)
            
                if "tree_sun" in label:
                    class_tree.append((col, row_px))
                    tree_x.append(col)
                    tree_y.append(row_px)
                elif "tree_shade" in label:
                    class_negative.append((col, row_px))
                    tree_shade_x.append(col)                
                    tree_shade_y.append(row_px)

                elif "green_wet_sun" in label or "green_dry_sun" in label:
                    class_other.append((col, row_px))
                    other_x.append(col)
                    other_y.append(row_px)


            img_np = np.asarray(img, dtype='float32') / 255

            fig, axs = plt.subplots(1, 5, figsize=(30, 10))
            axs[0].imshow(img_np)
            axs[0].scatter(tree_x, tree_y, c='red', marker='o', label='Tree_Sun')
            axs[0].scatter(tree_shade_x, tree_shade_y, c='purple', marker='o', label='Tree_Shade') 
            axs[0].scatter(other_x, other_y, c='blue', marker='x', label='Other')
            axs[0].legend()
            axs[0].set_title('Original Image with points')

            if class_tree:
                point_coords = np.array(class_tree)
                point_labels = np.ones(len(point_coords))

                if class_negative:
                    point_coords = np.concatenate((point_coords, np.array(class_negative)), axis=0)
                    point_labels = np.concatenate((point_labels, np.zeros(len(class_negative))))

                with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
                    predictor.set_image(img_np)
                    mask, _, _ = predictor.predict(point_coords=point_coords, point_labels=point_labels)
                mask = np.moveaxis(mask, 0, -1)

                mask1 = np.expand_dims(mask[:, :, 0], axis=-1)
                mask2 = np.expand_dims(mask[:, :, 1], axis=-1)
                mask3 = np.expand_dims(mask[:, :, 2], axis=-1)
                mask4 = np.expand_dims(mask1[:, :, 0] + mask2[:, :, 0] + mask3[:, :, 0], axis=-1)
                mask4[mask4 > 1] = 1

                out_image1 = np.multiply(mask1, img_np)
                out_image2 = np.multiply(mask2, img_np)
                out_image3 = np.multiply(mask3, img_np)
                out_image4 = np.multiply(mask4, img_np)

                axs[1].imshow(out_image1)
                axs[2].imshow(out_image2)
                axs[3].imshow(out_image3)
                axs[4].imshow(out_image4)

                axs[1].set_title('1st Channel Mask')
                axs[2].set_title('2nd Channel Mask')
                axs[3].set_title('3rd Channel Mask')
                axs[4].set_title('Combined Mask')

                Image.fromarray((out_image1 * 255).astype(np.uint8)).save(os.path.join(channel_paths['channel1'], filename))
                Image.fromarray((out_image2 * 255).astype(np.uint8)).save(os.path.join(channel_paths['channel2'], filename))
                Image.fromarray((out_image3 * 255).astype(np.uint8)).save(os.path.join(channel_paths['channel3'], filename))
                Image.fromarray((out_image4 * 255).astype(np.uint8)).save(os.path.join(channel_paths['combined'], filename))
            else:
                print("No 'Tree_Sun'-Points. Segmentation skipped.")

            plt.tight_layout()
            plt.savefig(os.path.join(results_path, filename) + '_overview.png')
            plt.show()


# ----------- Use all folders ------------------

base_dir = r"C:/Users/User/Documents/Bewaesserung_Forstkulturen/GH/Data/Thermal/GH_Analyse_Thermal"

thermal_folders = []
shapefile_folders = []

# Search for all relevant folders
for entry in os.listdir(base_dir):
    full_path = os.path.join(base_dir, entry)
    if os.path.isdir(full_path):
        if entry.endswith("_Thermal_sorted-richtig"):
            thermal_folders.append(full_path)
        elif entry.endswith("_Shapefiles"):
            shapefile_folders.append(full_path)

# Use the corresponding shapefile folder for every folder with thermal images
for thermal_folder in thermal_folders:
    match = re.search(r'20\d{6}', thermal_folder)
    date_str = match.group(0) if match else None
    if not date_str:
        print(f"No date found in foldername: {thermal_folder}")
        continue

    matching_shapes = [sf for sf in shapefile_folders if date_str in sf]
    if not matching_shapes:
        print(f"No shapefile folder found for date {date_str}")
        continue
    if len(matching_shapes) > 1:
        print(f"Several shapefile folder found for date {date_str}, use first one: {matching_shapes[0]}")

    shape_folder = matching_shapes[0]
    process_folder_pair(thermal_folder, shape_folder)


In [ ]:
# Repeat for Green_Wet_Sun

def world_to_pixel(transform, x, y):
    col, row = ~transform * (x, y)
    return int(col), int(row)

def process_folder_pair(folder_path, shape_path):
    processed_folder = os.path.join(folder_path, "Processed_new")
    if not os.path.exists(processed_folder):
        print(f"Processed_new folder not found in {folder_path}")
        return

    match = re.search(r'20\d{6}', folder_path)
    date_str = match.group(0) if match else 'unknown_date'

    base_results_dir = r"D:/Bewaesserung_Forstkulturen/Daten/Gewaechshaus/Gewaechshaus_Trockenstress2023/Thermal/GH_Analyse_Thermal/segmentation_results"
    folder_name_cleaned = os.path.basename(os.path.normpath(folder_path)).replace(date_str + "_", "")
    results_subdir = f'{date_str}_{folder_name_cleaned}_Green_Wet_Sun'
    results_path = os.path.join(base_results_dir, results_subdir)
    os.makedirs(results_path, exist_ok=True)

    checkpoint = "checkpoints/sam2.1_hiera_large.pt"
    model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"
    predictor = SAM2ImagePredictor(build_sam2(model_cfg, checkpoint))

    mask_channels = ['channel1', 'channel2', 'channel3', 'combined']
    channel_paths = {}
    for channel in mask_channels:
        path = os.path.join(results_path, channel)
        os.makedirs(path, exist_ok=True)
        channel_paths[channel] = path

    for filename in os.listdir(processed_folder):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            img_path = os.path.join(processed_folder, filename)
            print(f"Loading image: {img_path}")
            img = Image.open(img_path)

            basename = filename.replace("_thermal_rgb_transformed.jpg", "")
            matching_shapefiles = [f for f in os.listdir(shape_path) if f.startswith(basename) and f.endswith("_Points.shp")]

            if not matching_shapefiles:
                print(f"No shapefile found for basename: {basename}")
                continue
            elif len(matching_shapefiles) > 1:
                print(f"Multiple shapefiles found for basename {basename}, using the first: {matching_shapefiles[0]}")

            shape_name = os.path.join(shape_path, matching_shapefiles[0])
            gdf = gpd.read_file(shape_name)
            print(f"Number of points in the shapefile: {len(gdf)}")

            width, height = img.size
            xmin, xmax = 0, 480
            ymin, ymax = -640, 0
            xres = (xmax - xmin) / width
            yres = (ymax - ymin) / height
            transform = Affine.translation(xmin, ymax) * Affine.scale(xres, -yres)

            class_tree, class_other = [], []
            class_negative = []
            tree_x, tree_y = [], []
            other_x, other_y = [], []

            for _, row in gdf.iterrows():
                # Check for valid geometry
                if row.geometry is None or row.geometry.is_empty:
                    print("Skipped empty or invalid geometry.")
                    continue
                if row.geometry.geom_type != 'Point':
                    print(f"Unsupported geometry type: {row.geometry.geom_type}. Skipped.")
                    continue
            
                label = str(row["Position"]).strip().lower()
                x, y = row.geometry.x, row.geometry.y
                col, row_px = world_to_pixel(transform, x, y)
            
                if "green_wet_sun" in label:
                    class_tree.append((col, row_px))
                    tree_x.append(col)
                    tree_y.append(row_px)
                elif "green_wet_shade" in label:
                    class_negative.append((col, row_px))
                elif "tree_sun" in label or "green_dry_sun" in label:
                    class_other.append((col, row_px))
                    other_x.append(col)
                    other_y.append(row_px)


            img_np = np.asarray(img, dtype='float32') / 255

            fig, axs = plt.subplots(1, 5, figsize=(30, 10))
            axs[0].imshow(img_np)
            axs[0].scatter(tree_x, tree_y, c='red', marker='o', label='Green_Wet_Sun')
            axs[0].scatter(other_x, other_y, c='blue', marker='x', label='Other')
            axs[0].legend()
            axs[0].set_title('Original Image with points')

            if class_tree:
                point_coords = np.array(class_tree)
                point_labels = np.ones(len(point_coords))

                if class_negative:
                    point_coords = np.concatenate((point_coords, np.array(class_negative)), axis=0)
                    point_labels = np.concatenate((point_labels, np.zeros(len(class_negative))))

                with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
                    predictor.set_image(img_np)
                    mask, _, _ = predictor.predict(point_coords=point_coords, point_labels=point_labels)
                mask = np.moveaxis(mask, 0, -1)

                mask1 = np.expand_dims(mask[:, :, 0], axis=-1)
                mask2 = np.expand_dims(mask[:, :, 1], axis=-1)
                mask3 = np.expand_dims(mask[:, :, 2], axis=-1)
                mask4 = np.expand_dims(mask1[:, :, 0] + mask2[:, :, 0] + mask3[:, :, 0], axis=-1)
                mask4[mask4 > 1] = 1

                out_image1 = np.multiply(mask1, img_np)
                out_image2 = np.multiply(mask2, img_np)
                out_image3 = np.multiply(mask3, img_np)
                out_image4 = np.multiply(mask4, img_np)

                axs[1].imshow(out_image1)
                axs[2].imshow(out_image2)
                axs[3].imshow(out_image3)
                axs[4].imshow(out_image4)

                axs[1].set_title('1st Channel Mask')
                axs[2].set_title('2nd Channel Mask')
                axs[3].set_title('3rd Channel Mask')
                axs[4].set_title('Combined Mask')

                Image.fromarray((out_image1 * 255).astype(np.uint8)).save(os.path.join(channel_paths['channel1'], filename))
                Image.fromarray((out_image2 * 255).astype(np.uint8)).save(os.path.join(channel_paths['channel2'], filename))
                Image.fromarray((out_image3 * 255).astype(np.uint8)).save(os.path.join(channel_paths['channel3'], filename))
                Image.fromarray((out_image4 * 255).astype(np.uint8)).save(os.path.join(channel_paths['combined'], filename))
            else:
                print("No 'Green_Wet_Sun'-Points. Segmentation skipped.")

            plt.tight_layout()
            plt.savefig(os.path.join(results_path, filename) + '_overview.png')
            plt.show()


# ----------- Use all folders ------------------

base_dir = r"C:/Users/User/Documents/Bewaesserung_Forstkulturen/GH/Data/Thermal/GH_Analyse_Thermal"

thermal_folders = []
shapefile_folders = []

# Search for all relevant folders
for entry in os.listdir(base_dir):
    full_path = os.path.join(base_dir, entry)
    if os.path.isdir(full_path):
        if entry.endswith("_Thermal_sorted-richtig"):
            thermal_folders.append(full_path)
        elif entry.endswith("_Shapefiles"):
            shapefile_folders.append(full_path)

# Use the corresponding shapefile folder for every folder with thermal images
for thermal_folder in thermal_folders:
    match = re.search(r'20\d{6}', thermal_folder)
    date_str = match.group(0) if match else None
    if not date_str:
        print(f"No date found in foldername: {thermal_folder}")
        continue

    matching_shapes = [sf for sf in shapefile_folders if date_str in sf]
    if not matching_shapes:
        print(f"No shapefile folder found for date {date_str}")
        continue
    if len(matching_shapes) > 1:
        print(f"Several shapefile folder found for date {date_str}, use first one: {matching_shapes[0]}")

    shape_folder = matching_shapes[0]
    process_folder_pair(thermal_folder, shape_folder)


In [ ]:
# Repeat for Green_Dry_Sun

def world_to_pixel(transform, x, y):
    col, row = ~transform * (x, y)
    return int(col), int(row)

def process_folder_pair(folder_path, shape_path):
    processed_folder = os.path.join(folder_path, "Processed_new")
    if not os.path.exists(processed_folder):
        print(f"Processed_new folder not found in {folder_path}")
        return

    match = re.search(r'20\d{6}', folder_path)
    date_str = match.group(0) if match else 'unknown_date'

    base_results_dir = r"D:/Bewaesserung_Forstkulturen/Daten/Gewaechshaus/Gewaechshaus_Trockenstress2023/Thermal/GH_Analyse_Thermal/segmentation_results"
    folder_name_cleaned = os.path.basename(os.path.normpath(folder_path)).replace(date_str + "_", "")
    results_subdir = f'{date_str}_{folder_name_cleaned}_Green_Dry_Sun'
    results_path = os.path.join(base_results_dir, results_subdir)
    os.makedirs(results_path, exist_ok=True)

    checkpoint = "checkpoints/sam2.1_hiera_large.pt"
    model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"
    predictor = SAM2ImagePredictor(build_sam2(model_cfg, checkpoint))

    mask_channels = ['channel1', 'channel2', 'channel3', 'combined']
    channel_paths = {}
    for channel in mask_channels:
        path = os.path.join(results_path, channel)
        os.makedirs(path, exist_ok=True)
        channel_paths[channel] = path

    for filename in os.listdir(processed_folder):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            img_path = os.path.join(processed_folder, filename)
            print(f"Loading image: {img_path}")
            img = Image.open(img_path)

            basename = filename.replace("_thermal_rgb_transformed.jpg", "")
            matching_shapefiles = [f for f in os.listdir(shape_path) if f.startswith(basename) and f.endswith("_Points.shp")]

            if not matching_shapefiles:
                print(f"No shapefile found for basename: {basename}")
                continue
            elif len(matching_shapefiles) > 1:
                print(f"Multiple shapefiles found for basename {basename}, using the first: {matching_shapefiles[0]}")

            shape_name = os.path.join(shape_path, matching_shapefiles[0])
            gdf = gpd.read_file(shape_name)
            print(f"Number of points in the shapefile: {len(gdf)}")

            width, height = img.size
            xmin, xmax = 0, 480
            ymin, ymax = -640, 0
            xres = (xmax - xmin) / width
            yres = (ymax - ymin) / height
            transform = Affine.translation(xmin, ymax) * Affine.scale(xres, -yres)

            class_tree, class_other = [], []
            class_negative = []
            tree_x, tree_y = [], []
            other_x, other_y = [], []

            for _, row in gdf.iterrows():
                # Check for valid geometry
                if row.geometry is None or row.geometry.is_empty:
                    print("Skipped empty or invalid geometry.")
                    continue
                if row.geometry.geom_type != 'Point':
                    print(f"Unsupported geometry type: {row.geometry.geom_type}. Skipped.")
                    continue
            
                label = str(row["Position"]).strip().lower()
                x, y = row.geometry.x, row.geometry.y
                col, row_px = world_to_pixel(transform, x, y)
            
                if "green_dry_sun" in label:
                    class_tree.append((col, row_px))
                    tree_x.append(col)
                    tree_y.append(row_px)
                elif "green_dry_shade" in label:
                    class_negative.append((col, row_px))
                elif "tree_sun" in label or "green_wet_sun" in label:
                    class_other.append((col, row_px))
                    other_x.append(col)
                    other_y.append(row_px)


            img_np = np.asarray(img, dtype='float32') / 255

            fig, axs = plt.subplots(1, 5, figsize=(30, 10))
            axs[0].imshow(img_np)
            axs[0].scatter(tree_x, tree_y, c='red', marker='o', label='Green_Dry_Sun')
            axs[0].scatter(other_x, other_y, c='blue', marker='x', label='Other')
            axs[0].legend()
            axs[0].set_title('Original Image with points')

            if class_tree:
                point_coords = np.array(class_tree)
                point_labels = np.ones(len(point_coords))

                if class_negative:
                    point_coords = np.concatenate((point_coords, np.array(class_negative)), axis=0)
                    point_labels = np.concatenate((point_labels, np.zeros(len(class_negative))))

                with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
                    predictor.set_image(img_np)
                    mask, _, _ = predictor.predict(point_coords=point_coords, point_labels=point_labels)
                mask = np.moveaxis(mask, 0, -1)

                mask1 = np.expand_dims(mask[:, :, 0], axis=-1)
                mask2 = np.expand_dims(mask[:, :, 1], axis=-1)
                mask3 = np.expand_dims(mask[:, :, 2], axis=-1)
                mask4 = np.expand_dims(mask1[:, :, 0] + mask2[:, :, 0] + mask3[:, :, 0], axis=-1)
                mask4[mask4 > 1] = 1

                out_image1 = np.multiply(mask1, img_np)
                out_image2 = np.multiply(mask2, img_np)
                out_image3 = np.multiply(mask3, img_np)
                out_image4 = np.multiply(mask4, img_np)

                axs[1].imshow(out_image1)
                axs[2].imshow(out_image2)
                axs[3].imshow(out_image3)
                axs[4].imshow(out_image4)

                axs[1].set_title('1st Channel Mask')
                axs[2].set_title('2nd Channel Mask')
                axs[3].set_title('3rd Channel Mask')
                axs[4].set_title('Combined Mask')

                Image.fromarray((out_image1 * 255).astype(np.uint8)).save(os.path.join(channel_paths['channel1'], filename))
                Image.fromarray((out_image2 * 255).astype(np.uint8)).save(os.path.join(channel_paths['channel2'], filename))
                Image.fromarray((out_image3 * 255).astype(np.uint8)).save(os.path.join(channel_paths['channel3'], filename))
                Image.fromarray((out_image4 * 255).astype(np.uint8)).save(os.path.join(channel_paths['combined'], filename))
            else:
                print("No 'Green_Dry_Sun'-Points. Segmentation skipped.")

            plt.tight_layout()
            plt.savefig(os.path.join(results_path, filename) + '_overview.png')
            plt.show()


# ----------- Use all folders ------------------

base_dir = r"C:/Users/User/Documents/Bewaesserung_Forstkulturen/GH/Data/Thermal/GH_Analyse_Thermal"

thermal_folders = []
shapefile_folders = []

# Search for all relevant folders
for entry in os.listdir(base_dir):
    full_path = os.path.join(base_dir, entry)
    if os.path.isdir(full_path):
        if entry.endswith("_Thermal_sorted-richtig"):
            thermal_folders.append(full_path)
        elif entry.endswith("_Shapefiles"):
            shapefile_folders.append(full_path)

# Use the corresponding shapefile folder for every folder with thermal images
for thermal_folder in thermal_folders:
    match = re.search(r'20\d{6}', thermal_folder)
    date_str = match.group(0) if match else None
    if not date_str:
        print(f"No date found in foldername: {thermal_folder}")
        continue

    matching_shapes = [sf for sf in shapefile_folders if date_str in sf]
    if not matching_shapes:
        print(f"No shapefile folder found for date {date_str}")
        continue
    if len(matching_shapes) > 1:
        print(f"Several shapefile folder found for date {date_str}, use first one: {matching_shapes[0]}")

    shape_folder = matching_shapes[0]
    process_folder_pair(thermal_folder, shape_folder)


In [ ]:
# Repeat for Green_Wet_Shade

def world_to_pixel(transform, x, y):
    col, row = ~transform * (x, y)
    return int(col), int(row)

def process_folder_pair(folder_path, shape_path):
    processed_folder = os.path.join(folder_path, "Processed_new")
    if not os.path.exists(processed_folder):
        print(f"Processed_new folder not found in {folder_path}")
        return

    match = re.search(r'20\d{6}', folder_path)
    date_str = match.group(0) if match else 'unknown_date'

    base_results_dir = r"D:/Bewaesserung_Forstkulturen/Daten/Gewaechshaus/Gewaechshaus_Trockenstress2023/Thermal/GH_Analyse_Thermal/segmentation_results"
    folder_name_cleaned = os.path.basename(os.path.normpath(folder_path)).replace(date_str + "_", "")
    results_subdir = f'{date_str}_{folder_name_cleaned}_Green_Wet_Shade'
    results_path = os.path.join(base_results_dir, results_subdir)
    os.makedirs(results_path, exist_ok=True)

    checkpoint = "checkpoints/sam2.1_hiera_large.pt"
    model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"
    predictor = SAM2ImagePredictor(build_sam2(model_cfg, checkpoint))

    mask_channels = ['channel1', 'channel2', 'channel3', 'combined']
    channel_paths = {}
    for channel in mask_channels:
        path = os.path.join(results_path, channel)
        os.makedirs(path, exist_ok=True)
        channel_paths[channel] = path

    for filename in os.listdir(processed_folder):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            img_path = os.path.join(processed_folder, filename)
            print(f"Loading image: {img_path}")
            img = Image.open(img_path)

            basename = filename.replace("_thermal_rgb_transformed.jpg", "")
            matching_shapefiles = [f for f in os.listdir(shape_path) if f.startswith(basename) and f.endswith("_Points.shp")]

            if not matching_shapefiles:
                print(f"No shapefile found for basename: {basename}")
                continue
            elif len(matching_shapefiles) > 1:
                print(f"Multiple shapefiles found for basename {basename}, using the first: {matching_shapefiles[0]}")

            shape_name = os.path.join(shape_path, matching_shapefiles[0])
            gdf = gpd.read_file(shape_name)
            print(f"Number of points in the shapefile: {len(gdf)}")

            width, height = img.size
            xmin, xmax = 0, 480
            ymin, ymax = -640, 0
            xres = (xmax - xmin) / width
            yres = (ymax - ymin) / height
            transform = Affine.translation(xmin, ymax) * Affine.scale(xres, -yres)

            class_tree, class_other = [], []
            class_negative = []
            tree_x, tree_y = [], []
            other_x, other_y = [], []

            for _, row in gdf.iterrows():
                # Check for valid geometry
                if row.geometry is None or row.geometry.is_empty:
                    print("Skipped empty or invalid geometry.")
                    continue
                if row.geometry.geom_type != 'Point':
                    print(f"Unsupported geometry type: {row.geometry.geom_type}. Skipped.")
                    continue
            
                label = str(row["Position"]).strip().lower()
                x, y = row.geometry.x, row.geometry.y
                col, row_px = world_to_pixel(transform, x, y)
            
                if "green_wet_shade" in label:
                    class_tree.append((col, row_px))
                    tree_x.append(col)
                    tree_y.append(row_px)
                elif "green_wet_sun" in label:
                    class_negative.append((col, row_px))
                elif "tree_shade" in label or "green_dry_shade" in label:
                    class_other.append((col, row_px))
                    other_x.append(col)
                    other_y.append(row_px)


            img_np = np.asarray(img, dtype='float32') / 255

            fig, axs = plt.subplots(1, 5, figsize=(30, 10))
            axs[0].imshow(img_np)
            axs[0].scatter(tree_x, tree_y, c='red', marker='o', label='Green_Wet_Shade')
            axs[0].scatter(other_x, other_y, c='blue', marker='x', label='Other')
            axs[0].legend()
            axs[0].set_title('Original Image with points')

            if class_tree:
                point_coords = np.array(class_tree)
                point_labels = np.ones(len(point_coords))

                if class_negative:
                    point_coords = np.concatenate((point_coords, np.array(class_negative)), axis=0)
                    point_labels = np.concatenate((point_labels, np.zeros(len(class_negative))))

                with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
                    predictor.set_image(img_np)
                    mask, _, _ = predictor.predict(point_coords=point_coords, point_labels=point_labels)
                mask = np.moveaxis(mask, 0, -1)

                mask1 = np.expand_dims(mask[:, :, 0], axis=-1)
                mask2 = np.expand_dims(mask[:, :, 1], axis=-1)
                mask3 = np.expand_dims(mask[:, :, 2], axis=-1)
                mask4 = np.expand_dims(mask1[:, :, 0] + mask2[:, :, 0] + mask3[:, :, 0], axis=-1)
                mask4[mask4 > 1] = 1

                out_image1 = np.multiply(mask1, img_np)
                out_image2 = np.multiply(mask2, img_np)
                out_image3 = np.multiply(mask3, img_np)
                out_image4 = np.multiply(mask4, img_np)

                axs[1].imshow(out_image1)
                axs[2].imshow(out_image2)
                axs[3].imshow(out_image3)
                axs[4].imshow(out_image4)

                axs[1].set_title('1st Channel Mask')
                axs[2].set_title('2nd Channel Mask')
                axs[3].set_title('3rd Channel Mask')
                axs[4].set_title('Combined Mask')

                Image.fromarray((out_image1 * 255).astype(np.uint8)).save(os.path.join(channel_paths['channel1'], filename))
                Image.fromarray((out_image2 * 255).astype(np.uint8)).save(os.path.join(channel_paths['channel2'], filename))
                Image.fromarray((out_image3 * 255).astype(np.uint8)).save(os.path.join(channel_paths['channel3'], filename))
                Image.fromarray((out_image4 * 255).astype(np.uint8)).save(os.path.join(channel_paths['combined'], filename))
            else:
                print("No 'Green_Wet_Shade'-Points. Segmentation skipped.")

            plt.tight_layout()
            plt.savefig(os.path.join(results_path, filename) + '_overview.png')
            plt.show()


# ----------- Use all folders ------------------

base_dir = r"C:/Users/User/Documents/Bewaesserung_Forstkulturen/GH/Data/Thermal/GH_Analyse_Thermal"

thermal_folders = []
shapefile_folders = []

# Search for all relevant folders
for entry in os.listdir(base_dir):
    full_path = os.path.join(base_dir, entry)
    if os.path.isdir(full_path):
        if entry.endswith("_Thermal_sorted-richtig"):
            thermal_folders.append(full_path)
        elif entry.endswith("_Shapefiles"):
            shapefile_folders.append(full_path)

# Use the corresponding shapefile folder for every folder with thermal images
for thermal_folder in thermal_folders:
    match = re.search(r'20\d{6}', thermal_folder)
    date_str = match.group(0) if match else None
    if not date_str:
        print(f"No date found in foldername: {thermal_folder}")
        continue

    matching_shapes = [sf for sf in shapefile_folders if date_str in sf]
    if not matching_shapes:
        print(f"No shapefile folder found for date {date_str}")
        continue
    if len(matching_shapes) > 1:
        print(f"Several shapefile folder found for date {date_str}, use first one: {matching_shapes[0]}")

    shape_folder = matching_shapes[0]
    process_folder_pair(thermal_folder, shape_folder)


In [ ]:
# Repeat for Green_Dry_Shade

def world_to_pixel(transform, x, y):
    col, row = ~transform * (x, y)
    return int(col), int(row)

def process_folder_pair(folder_path, shape_path):
    processed_folder = os.path.join(folder_path, "Processed_new")
    if not os.path.exists(processed_folder):
        print(f"Processed_new folder not found in {folder_path}")
        return

    match = re.search(r'20\d{6}', folder_path)
    date_str = match.group(0) if match else 'unknown_date'

    base_results_dir = r"D:/Bewaesserung_Forstkulturen/Daten/Gewaechshaus/Gewaechshaus_Trockenstress2023/Thermal/GH_Analyse_Thermal/segmentation_results"
    folder_name_cleaned = os.path.basename(os.path.normpath(folder_path)).replace(date_str + "_", "")
    results_subdir = f'{date_str}_{folder_name_cleaned}_Green_Dry_Shade'
    results_path = os.path.join(base_results_dir, results_subdir)
    os.makedirs(results_path, exist_ok=True)

    checkpoint = "checkpoints/sam2.1_hiera_large.pt"
    model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"
    predictor = SAM2ImagePredictor(build_sam2(model_cfg, checkpoint))

    mask_channels = ['channel1', 'channel2', 'channel3', 'combined']
    channel_paths = {}
    for channel in mask_channels:
        path = os.path.join(results_path, channel)
        os.makedirs(path, exist_ok=True)
        channel_paths[channel] = path

    for filename in os.listdir(processed_folder):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            img_path = os.path.join(processed_folder, filename)
            print(f"Loading image: {img_path}")
            img = Image.open(img_path)

            basename = filename.replace("_thermal_rgb_transformed.jpg", "")
            matching_shapefiles = [f for f in os.listdir(shape_path) if f.startswith(basename) and f.endswith("_Points.shp")]

            if not matching_shapefiles:
                print(f"No shapefile found for basename: {basename}")
                continue
            elif len(matching_shapefiles) > 1:
                print(f"Multiple shapefiles found for basename {basename}, using the first: {matching_shapefiles[0]}")

            shape_name = os.path.join(shape_path, matching_shapefiles[0])
            gdf = gpd.read_file(shape_name)
            print(f"Number of points in the shapefile: {len(gdf)}")

            width, height = img.size
            xmin, xmax = 0, 480
            ymin, ymax = -640, 0
            xres = (xmax - xmin) / width
            yres = (ymax - ymin) / height
            transform = Affine.translation(xmin, ymax) * Affine.scale(xres, -yres)

            class_tree, class_other = [], []
            class_negative = []
            tree_x, tree_y = [], []
            other_x, other_y = [], []

            for _, row in gdf.iterrows():
                # Check for valid geometry
                if row.geometry is None or row.geometry.is_empty:
                    print("Skipped empty or invalid geometry.")
                    continue
                if row.geometry.geom_type != 'Point':
                    print(f"Unsupported geometry type: {row.geometry.geom_type}. Skipped.")
                    continue
            
                label = str(row["Position"]).strip().lower()
                x, y = row.geometry.x, row.geometry.y
                col, row_px = world_to_pixel(transform, x, y)
            
                if "green_dry_shade" in label:
                    class_tree.append((col, row_px))
                    tree_x.append(col)
                    tree_y.append(row_px)
                elif "green_dry_sun" in label:
                    class_negative.append((col, row_px))
                elif "tree_shade" in label or "green_wet_shade" in label:
                    class_other.append((col, row_px))
                    other_x.append(col)
                    other_y.append(row_px)


            img_np = np.asarray(img, dtype='float32') / 255

            fig, axs = plt.subplots(1, 5, figsize=(30, 10))
            axs[0].imshow(img_np)
            axs[0].scatter(tree_x, tree_y, c='red', marker='o', label='Green_Dry_Shade')
            axs[0].scatter(other_x, other_y, c='blue', marker='x', label='Other')
            axs[0].legend()
            axs[0].set_title('Original Image with points')

            if class_tree:
                point_coords = np.array(class_tree)
                point_labels = np.ones(len(point_coords))

                if class_negative:
                    point_coords = np.concatenate((point_coords, np.array(class_negative)), axis=0)
                    point_labels = np.concatenate((point_labels, np.zeros(len(class_negative))))

                with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
                    predictor.set_image(img_np)
                    mask, _, _ = predictor.predict(point_coords=point_coords, point_labels=point_labels)
                mask = np.moveaxis(mask, 0, -1)

                mask1 = np.expand_dims(mask[:, :, 0], axis=-1)
                mask2 = np.expand_dims(mask[:, :, 1], axis=-1)
                mask3 = np.expand_dims(mask[:, :, 2], axis=-1)
                mask4 = np.expand_dims(mask1[:, :, 0] + mask2[:, :, 0] + mask3[:, :, 0], axis=-1)
                mask4[mask4 > 1] = 1

                out_image1 = np.multiply(mask1, img_np)
                out_image2 = np.multiply(mask2, img_np)
                out_image3 = np.multiply(mask3, img_np)
                out_image4 = np.multiply(mask4, img_np)

                axs[1].imshow(out_image1)
                axs[2].imshow(out_image2)
                axs[3].imshow(out_image3)
                axs[4].imshow(out_image4)

                axs[1].set_title('1st Channel Mask')
                axs[2].set_title('2nd Channel Mask')
                axs[3].set_title('3rd Channel Mask')
                axs[4].set_title('Combined Mask')

                Image.fromarray((out_image1 * 255).astype(np.uint8)).save(os.path.join(channel_paths['channel1'], filename))
                Image.fromarray((out_image2 * 255).astype(np.uint8)).save(os.path.join(channel_paths['channel2'], filename))
                Image.fromarray((out_image3 * 255).astype(np.uint8)).save(os.path.join(channel_paths['channel3'], filename))
                Image.fromarray((out_image4 * 255).astype(np.uint8)).save(os.path.join(channel_paths['combined'], filename))
            else:
                print("No 'Green_Dry_Shade'-Points. Segmentation skipped.")

            plt.tight_layout()
            plt.savefig(os.path.join(results_path, filename) + '_overview.png')
            plt.show()


# ----------- Use all folders ------------------

base_dir = r"C:/Users/User/Documents/Bewaesserung_Forstkulturen/GH/Data/Thermal/GH_Analyse_Thermal"

thermal_folders = []
shapefile_folders = []

# Search for all relevant folders
for entry in os.listdir(base_dir):
    full_path = os.path.join(base_dir, entry)
    if os.path.isdir(full_path):
        if entry.endswith("_Thermal_sorted-richtig"):
            thermal_folders.append(full_path)
        elif entry.endswith("_Shapefiles"):
            shapefile_folders.append(full_path)

# Use the corresponding shapefile folder for every folder with thermal images
for thermal_folder in thermal_folders:
    match = re.search(r'20\d{6}', thermal_folder)
    date_str = match.group(0) if match else None
    if not date_str:
        print(f"No date found in foldername: {thermal_folder}")
        continue

    matching_shapes = [sf for sf in shapefile_folders if date_str in sf]
    if not matching_shapes:
        print(f"No shapefile folder found for date {date_str}")
        continue
    if len(matching_shapes) > 1:
        print(f"Several shapefile folder found for date {date_str}, use first one: {matching_shapes[0]}")

    shape_folder = matching_shapes[0]
    process_folder_pair(thermal_folder, shape_folder)


In [2]:
# check if all files have been created

base_dir = r"D:\Data\Backup\Bewaesserung_Forstkulturen\GH\GH_Analysis\Thermal_Image_Processing\segmentation_results"

# Helper function for counting imagefiles in folder
def count_files(folder):
    return len([f for f in os.listdir(folder) if os.path.isfile(os.path.join(folder, f))])

# group folders according to  dates
date_groups = defaultdict(list)

for folder in os.listdir(base_dir):
    folder_path = os.path.join(base_dir, folder)
    if os.path.isdir(folder_path):
        match = re.match(r'^(20\d{6})', folder)
        if match:
            date = match.group(1)
            date_groups[date].append(folder_path)

# Compare fileamoungs within dategroups
for date, folders in date_groups.items():
    file_counts = {folder: count_files(folder) for folder in folders}
    unique_counts = set(file_counts.values())
    
    if len(unique_counts) > 1:
        print(f"\n❌ Amount of files differs for date {date}:")
        for folder, count in file_counts.items():
            print(f"  {os.path.basename(folder)} → {count} files")
    else:
        print(f"✔ {date}: All {len(folders)} folders have {unique_counts.pop()} files.")


✔ 20230627: All 6 folders have 80 files.
✔ 20230629: All 6 folders have 80 files.
✔ 20230704: All 6 folders have 80 files.
✔ 20230713: All 6 folders have 80 files.
✔ 20230718: All 6 folders have 80 files.
✔ 20230720: All 6 folders have 80 files.
✔ 20230725: All 6 folders have 80 files.
✔ 20230731: All 6 folders have 80 files.
✔ 20230803: All 6 folders have 79 files.
✔ 20230808: All 6 folders have 80 files.
✔ 20230811: All 6 folders have 22 files.
✔ 20230818: All 6 folders have 60 files.


In [3]:

thermal_folder = r"D:/Bewaesserung_Forstkulturen/Daten/Gewaechshaus/Gewaechshaus_Trockenstress2023/Thermal/GH_Analyse_Thermal/20230731_Thermal_sorted/Processed_new"
shapefile_folder = r"D:/Bewaesserung_Forstkulturen/Daten/Gewaechshaus/Gewaechshaus_Trockenstress2023/Thermal/GH_Analyse_Thermal/20230731_Shapefiles"

# Take image file according to suffix
expected_suffix = "_thermal_rgb_transformed.jpg"
image_files = [f for f in os.listdir(thermal_folder) if f.endswith(expected_suffix)]

# Take shapefiles with suffix "_Points.shp"
shapefile_files = [f for f in os.listdir(shapefile_folder) if f.endswith("_Points.shp")]

# Extract IDs
image_ids = [f.replace(expected_suffix, "") for f in image_files]
shapefile_ids = [f.replace("_Points.shp", "") for f in shapefile_files]

# Identify missing shapefiles
missing_shapefiles = set(image_ids) - set(shapefile_ids)

print("Amount images:", len(image_ids))
print("Amount shapefiles:", len(shapefile_ids))
print("\nMissing shapefiles for the following Tree-IDs:")
for missing_id in sorted(missing_shapefiles):
    print(missing_id)


Amount images: 80
Amount shapefiles: 80

Missing shapefiles for the following Tree-IDs:
